# 🔬 Notebook 3: Reddit — Deep Dive: Hot ranking, sharded counters, comment trees

## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — the "Hot" ranking formula

Reddit's historical formula (paraphrased):
```
score = log10(max(|ups - downs|, 1)) + sign(ups - downs) * age_seconds / 45000
```

Properties:
- log-scaling so a post with 10,000 votes isn't 10× above one with 1,000.
- **age pushes things up** (more recent = higher) rather than decaying old posts.
- Net-downvoted posts get a *negative* time bonus → they sink quickly.


In [ ]:
import math, time

def hot_score(ups: int, downs: int, age_seconds: float) -> float:
    net = ups - downs
    s = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    return round(order + s * age_seconds / 45000, 4)

now_age = 0  # brand new
# two posts posted now
print("Brand new, +10:", hot_score(10, 0, 0))
print("1 hour old, +10:", hot_score(10, 0, 3600))
print("Brand new, -5:", hot_score(0, 5, 0))
print("6 hour old, +1000:", hot_score(1000, 0, 6*3600))

# Ranking sanity check
posts = [
    ("just-posted/10up",     hot_score(10,   0, 0)),
    ("1h-old/1000up",        hot_score(1000, 0, 3600)),
    ("2h-old/200up/50down",  hot_score(200, 50, 2*3600)),
    ("24h-old/10000up",      hot_score(10000,0, 24*3600)),
    ("5min-old/5up",         hot_score(5,    0, 300)),
]
for name, s in sorted(posts, key=lambda x: -x[1]):
    print(f"  {s:>8.4f}  {name}")


## Deep dive 2 — vote fanout without hot-key contention

Problem: a mega-thread has millions of votes on one post. Incrementing a single row
`UPDATE posts SET ups = ups + 1 WHERE id = X` becomes the bottleneck.

**Fix: sharded counters.**

```
  votes_shard_0, votes_shard_1, ... votes_shard_63

  on vote:  UPDATE votes_shard_{uid mod 64} SET ups=ups+1 WHERE post_id=X
  on read:  SELECT SUM(ups), SUM(downs) FROM votes_shard_* WHERE post_id=X
```

Writes are distributed across 64 rows; reads are 64× more expensive but cached.


In [ ]:
# Tiny sharded counter simulation
class ShardedCounter:
    def __init__(self, n_shards=16):
        self.shards = [0] * n_shards
    def inc(self, user_id):
        self.shards[user_id % len(self.shards)] += 1
    def total(self):
        return sum(self.shards)

c = ShardedCounter(16)
for uid in range(100_000):
    c.inc(uid)
print("total votes:", c.total(), "distribution:", c.shards[:4], "...")


## Deep dive 3 — comment trees

Reddit comments are a *tree* per post. Loading a huge tree at once is slow; we need:
- **Collapsed-by-default** deep threads.
- **"Load more children"** button (pagination within a branch).
- Threaded sort orders: best / top / new / controversial.

### Storage
A simple approach: store `parent_id` on each comment, plus a `path` ("/3/47/51") for cheap subtree retrieval.
Materialized paths let you fetch a subtree with `WHERE path LIKE '/3/47/%'`.
